# SNR study of the fast-photometry test-star observations

Measured signal-to-noise ratio of every test-star observation, obtained in the
different ways explored in the old `check_fast_photometry_data.ipynb`, compared to the
theoretical SNR of `Instrument.noise_budget`, together with everything about each
observation that the binary files do not record and that could explain the disparity.

The pipeline itself (conditions, NSB, measured and theoretical SNR) lives in
`kbo_occultation.analysis.snr` and is what `kbo-night snr` runs; this notebook calls it
and adds the diagnostics.

**Sources**
- `kbo_occultation/data/observations/observation_log.yaml`: stars, magnitudes (SIMBAD B and V), coordinates, good-time windows, faulty mirrors, eLog notes. *Edit it there, not here.*
- `kbo_occultation/data/observations/conditions/<night>/`: weather, pyrometer, AMC, starguider and CC pointing, extracted from the MAGIC CC `.rep` files by `kbo-night prepare`.
- `DCs/<night>/dc_report.pkl`: slow DC reports (needs `magic_spysii` to unpickle).
- Zd, airmass and moon are computed with astropy from the coordinates and the run times, and cross-checked against the CC reports.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

from kbo_occultation import config
from kbo_occultation.observations.log import load_observation_log, nights_table, observations_table, validate
from kbo_occultation.observations.cc_reports import load_conditions, pointing_conditions, conditions_table
from kbo_occultation.simulation.detectability import bin_average
from kbo_occultation.analysis.snr import (N_BINS, DETREND_WINDOW_S, load_lightcurves, nsb_table,
                                          measured_snr_table, theoretical_snr_table, plot_snr_vs_magnitude)
from kbo_occultation.analysis.snr_measurement import (good_time_lightcurve, detrended_snr, band_noise_rms,
                                                      noise_vs_bin_size, frame_noise, variance_estimator_floor)

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 3)

log = load_observation_log()
NIGHT_COLORS = dict(zip(log.nights, ["tab:blue", "tab:red", "tab:green"]))
TELESCOPES = {ch: config.ORM_ARRAY.by_channel(ch) for ch in "ABC"}
MARKERS = {"MAGIC-1": "s", "MAGIC-2": "o", "LST-1": "^"}

## 1. Observation log

In [ ]:
nights_table(log)

In [ ]:
observations_table(log)

Faulty mirrors and eLog notes per night (as typed in the YAML), and everything still missing (`validate`, the same list `kbo-night prepare` prints), i.e. things to fill in from the eLog / shift reports:

In [ ]:
for night in log.nights.values():
    print(f"── {night.name}")
    print(f"   faulty mirrors: {night.faulty_mirrors}")
    print(f"   eLog: {night.elog_notes}")
    for problem in validate(night):
        print(f"   missing: {problem}")

## 2. Observing conditions

Per observation: pointing and moon (astropy), and the CC-report weather, pyrometer, starguider and AMC panel status averaged over the run. The CC-report Zd is shown as a cross-check of the computed one (CC wrote `00.00` all through 2025-12-16, so it is missing there). Weather and the pyrometer are site-wide; AMC is per telescope. A night's YAML `manual_conditions` (optional) are added for anything the reports don't cover.

In [ ]:
conditions = conditions_table(log)
conditions

In [ ]:
for night in log.nights.values():
    cond = load_conditions(night.conditions_dir) if night.conditions_dir else {}
    if not cond:
        continue
    site = cond.get("m2") or next(iter(cond.values()))
    spans = {o.name: o.time_span() for o in night.observations.values()}
    fig, axes = plt.subplots(5, 1, figsize=(12, 11), sharex=True)
    to_dt = lambda t: Time(t, format="unix").datetime
    cc = site["cc"]
    axes[0].plot(to_dt(cc.time), cc.zd_deg, ".", ms=2, label="CC Zd")
    for name, (t0, t1) in spans.items():
        obs = night.observations[name]
        if obs.ra_deg is not None:
            tt = np.linspace(t0, t1, 10)
            zd = [pointing_conditions(obs.ra_deg, obs.dec_deg, t, t + 1, 2)["zd_mean_deg"] for t in tt]
            axes[0].plot(to_dt(tt), zd, "k-", lw=2)
    axes[0].set_ylabel("Zd [deg]"); axes[0].legend(["CC report", "computed (target)"])
    w = site["weather"]
    axes[1].plot(to_dt(w.time), w.humidity_pct, label="humidity [%]")
    axes[1].plot(to_dt(w.time), w.wind_avg_kmh, label="wind avg [km/h]")
    axes[1].plot(to_dt(w.time), w.wind_peak_kmh, label="wind peak [km/h]", alpha=0.6)
    axes[1].legend(fontsize=8)
    p = site["pyro"]
    axes[2].plot(to_dt(p.time), p.cloudiness_pct, label="cloudiness [%]")
    ax2 = axes[2].twinx()
    ax2.plot(to_dt(p.time), p.sky_temperature_K - 273.15, color="tab:orange", label="sky T [C]")
    ax2.plot(to_dt(w.time), w.temperature_C, color="tab:red", ls="--", label="air T [C]")
    axes[2].set_ylabel("cloudiness [%]"); ax2.set_ylabel("T [C]"); ax2.legend(fontsize=8, loc="center right")
    for tel, reports in cond.items():
        a = reports["amc"]
        axes[3].plot(to_dt(a.time), a.amc_panels_1, label=f"{tel} PANELS[1]")
        axes[3].plot(to_dt(a.time), a.amc_panels_3, ls="--", label=f"{tel} PANELS[3]")
    axes[3].set_ylabel("AMC panel counts"); axes[3].legend(fontsize=8)
    for tel, reports in cond.items():
        s = reports["starg"]
        axes[4].plot(to_dt(s.time), s.starg_n_stars, ".", ms=2, label=f"{tel} starguider stars")
    axes[4].set_ylabel("# stars"); axes[4].legend(fontsize=8)
    for ax in axes:
        for name, (t0, t1) in spans.items():
            ax.axvspan(*to_dt([t0, t1]), color="grey", alpha=0.2)
    for name, (t0, t1) in spans.items():
        axes[0].text(to_dt(t0), axes[0].get_ylim()[1], name, rotation=90, va="top", fontsize=8)
    fig.suptitle(f"Observing conditions {night.name} (grey: fast-photometry runs)")
    fig.tight_layout()
    plt.show()

## 3. Light curves

Outlier-cleaned variance (`.npz` caches), binned to ~1 s for display. The shaded band is the good-time window from the log (the old notebook's `gt` dict); runs without one use the whole run.

In [ ]:
lcs = load_lightcurves(log)   # (night, observation, channel) -> LightCurve

for night in log.nights.values():
    fig, axes = plt.subplots(len(night.observations), len(night.channels), squeeze=False,
                             figsize=(5 * len(night.channels), 2.2 * len(night.observations)), sharex=True)
    for i, obs in enumerate(night.observations.values()):
        for j, ch in enumerate(night.channels):
            lc = lcs[(night.name, obs.name, ch)]
            t, s, _ = bin_average(lc.time - lc.time[0], lc.signal, int(round(1 / lc.dt)))
            ax = axes[i, j]
            ax.plot(t, s, lw=0.8)
            if obs.good_time:
                ax.axvspan(*obs.good_time, color="green", alpha=0.2)
            ax.set_title(f"{obs.name} – {TELESCOPES[ch].name}", fontsize=9)
    for ax in axes[-1]:
        ax.set_xlabel("time since start [s]")
    fig.suptitle(f"{night.name}: variance (std²), 1 s bins")
    fig.tight_layout()
    plt.show()

## 4. NSB (+ electronic pedestal) level

Three estimators, per observation and telescope:
- **dark**: mean of the night's dark run / dark patch in the fast signal (fast units). 2026-03-08 has no dark run.
- **dc_bkg**: the lower of the two DC background pixels during the run (DC units), converted to fast units with `dc_to_fast_scale` (mean fast / mean on-target DC).
- **dc_hist**: first peak of the on-target pixel's DC histogram over the whole report (meant for LST-1, whose background pixel reads high), converted the same way.

The default NSB used below is `dark` when available, otherwise `dc_hist` for LST-1 and `dc_bkg` for MAGIC.

In [ ]:
nsb = nsb_table(log, n_bin=N_BINS[0], lcs=lcs)
nsb

## 5. Measured SNR

All SNRs are at the binned cadence (`n_bin` samples per point) on the good-time window, with the default NSB of section 4.

| column | signal | noise | old notebook |
|---|---|---|---|
| `mean_fast` | mean of binned fast signal − NSB | ⟨per-bin std/√n⟩ ⊕ NSB error | method 1 |
| `peak_fast` | last histogram peak − NSB | same | method p1 |
| `mean_comb`, `peak_comb` | same on the fast+DC combined signal (DC units, NSB in DC units) | same | methods 2/3, p2/p3 |
| `mean_fast_dcnsb` | mean − NSB from the DC background pixel | same | method 3 |
| `detr` | baseline − NSB | robust bin-to-bin σ after removing a `DETREND_WINDOW_S` rolling baseline | (new) |
| `detr_full` | as `detr`, whole run instead of good-time window | | |
| `detr_despiked` | as `detr`, after dropping samples near DC excursions (wind/tracking dips) | | `remove_valleys_dc` |
| `detr_dcdetrend` | as `detr`, after dividing by the smoothed DC trend | | |
| `*_ext` | × extinction factor 10^(0.4 k_B (X−1)) | | Zd correction |

`mean_*`/`peak_*` only see scatter *within* a bin, so noise correlated over ≥ one bin is invisible to them and they run high (plus an ~8 % small-sample bias at n_bin=10). `detr` sees the bin-to-bin scatter, which is what a dip of that duration has to beat: **`detr` is the one to compare to theory.**

In [ ]:
snr = measured_snr_table(log, n_bins=N_BINS, nsb=nsb, conditions=conditions, lcs=lcs)
snr

## 6. Theoretical SNR

`Instrument.realistic_signal_to_noise_ratio` at each observation's binning, for the star's B and V magnitude (the model normalises to Vega over the unfiltered QE band, so neither is exact; for these late-B stars B−V ≈ 0 and the difference is small):

- `theo_shot`: shot noise only (old optimistic model);
- `theo_zenith`: shot × F² + electronic + scintillation at the zenith;
- `theo_obs`: same at the observation's airmass (scintillation **and** extinction), times the fraction of mirror area left after the faulty panels listed in the log.

The faulty panels are the observation's own `faulty_mirrors` entry when the log has one, otherwise the night's value.

In [ ]:
theory = theoretical_snr_table(snr, log, conditions)
results = snr.join(theory)
results["ratio_detr_B"] = results["detr"] / results["theo_obs_B"]
results["ratio_mean_B"] = results["mean_fast"] / results["theo_obs_B"]
results[["detr", "mean_fast", "theo_shot_B", "theo_zenith_B", "theo_obs_B", "ratio_detr_B", "ratio_mean_B",
         "airmass", "mirror_throughput"]]

## 7. Measured vs theoretical SNR

In [ ]:
for n in N_BINS:
    plot_snr_vs_magnitude(results, log, n)
    plt.show()

### Which approach moves the measurement closest to theory?
Median measured/theory (`theo_obs_B`) per night, telescope and approach:

In [ ]:
approaches = ["mean_fast", "peak_fast", "mean_comb", "peak_comb", "mean_fast_dcnsb", "detr", "detr_full",
              "detr_despiked", "detr_dcdetrend", "detr_ext"]
ratios = results[approaches].div(results["theo_obs_B"], axis=0)
ratios.groupby(level=["night", "telescope", "n_bin"]).median()

### Disparity vs observing conditions
Ratio of the detrended SNR to `theo_obs_B` (5 ms bins) against each candidate cause. A clear trend points at a cause the model does not include (or includes wrongly).

In [ ]:
n = N_BINS[0]
df = results.xs(n, level="n_bin").reset_index().merge(conditions.reset_index(), on=["night", "observation"])
df = df.merge(nsb.reset_index()[["night", "observation", "telescope", "nsb_default", "star_mean"]],
              on=["night", "observation", "telescope"])
df["nsb_over_star"] = df["nsb_default"] / (df["star_mean"] - df["nsb_default"])
candidates = ["zd_mean_deg", "humidity_pct", "cloudiness_pct", "sky_temperature_K", "wind_avg_kmh",
              "mirror_throughput", "nsb_over_star", "dc_bad_frac", "starg_n_stars"]
fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharey=True)
for ax, col in zip(axes.flat, candidates):
    for (night_name, tel_name), g in df.groupby(["night", "telescope"]):
        if col in g:
            ax.scatter(g[col], g["ratio_detr_B"], color=NIGHT_COLORS[night_name], marker=MARKERS[tel_name],
                       label=f"{night_name} {tel_name}")
    ax.set_xlabel(col); ax.set_yscale("log"); ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("detrended / theory"); axes[1, 0].set_ylabel("detrended / theory"); axes[2, 0].set_ylabel("detrended / theory")
axes[0, 0].legend(fontsize=7)
fig.tight_layout()
plt.show()

## 8. Noise diagnostics

Fractional noise vs bin size, within-bin (open) and bin-to-bin after detrending (filled), against the white-noise 1/√n expectation (dashed, anchored at the smallest bin). Where the filled curve flattens away from the dashed line, correlated noise dominates and extra averaging stops helping. The good-time windows are short (30–60 s) for 2025-12-16 and 2026-03-08, so the whole run is used here. Below: band-limited RMS of the fractional flux from the PSD (the old notebook's band-RMS cells).

In [ ]:
bin_sizes = np.unique(np.round(np.logspace(np.log10(2), 3.5, 20)).astype(int))
fig, axes = plt.subplots(1, len(log.nights), figsize=(6 * len(log.nights), 5), sharey=True)
for ax, night in zip(axes, log.nights.values()):
    for obs in night.stars.values():
        lc = lcs[(night.name, obs.name, "A")]
        nsb0 = nsb.loc[(night.name, obs.name, "MAGIC-2"), "nsb_default"]
        n_used, within, between = noise_vs_bin_size(lc, bin_sizes, nsb0, DETREND_WINDOW_S)
        bin_ms = n_used * lc.dt * 1e3
        line, = ax.loglog(bin_ms, between, "o-", ms=3, label=f"{obs.name} (B={obs.mag_B})")
        ax.loglog(bin_ms, within, "o", ms=3, mfc="none", color=line.get_color(), alpha=0.6)
        ax.loglog(bin_ms, between[0] / np.sqrt(n_used / n_used[0]), ls="--", color=line.get_color(), alpha=0.5)
    ax.set_title(f"{night.name} MAGIC-2"); ax.set_xlabel("bin [ms]"); ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)
axes[0].set_ylabel("fractional noise per bin")
fig.tight_layout()
plt.show()

In [ ]:
PSD_BANDS = {"0.1-1 Hz": (0.1, 1), "1-10 Hz": (1, 10), "10-100 Hz": (10, 100), "100-900 Hz": (100, 900)}
psd_rows = []
for night in log.nights.values():
    for ch in night.channels:
        for obs in night.observations.values():
            lc = lcs[(night.name, obs.name, ch)]
            psd_rows.append({"night": night.name, "observation": obs.name, "telescope": TELESCOPES[ch].name,
                             **band_noise_rms(lc, PSD_BANDS, nperseg=2**14)})
pd.DataFrame(psd_rows).set_index(["night", "observation", "telescope"])

### How much of the noise does the detrend window remove?
Detrended SNR (5 ms bins, good-time window) for several baseline windows. A strong dependence means the noise lives at frequencies around 1/window, i.e. it is slow variability rather than per-sample noise.

In [ ]:
windows_s = [0.5, 2.0, 10.0, 30.0]
win_rows = []
for night in log.nights.values():
    for ch in night.channels:
        for obs in night.stars.values():
            lc = good_time_lightcurve(lcs[(night.name, obs.name, ch)], obs.good_time)
            nsb0 = nsb.loc[(night.name, obs.name, TELESCOPES[ch].name), "nsb_default"]
            win_rows.append({"night": night.name, "observation": obs.name, "telescope": TELESCOPES[ch].name,
                             **{f"detr_{w:g}s": detrended_snr(lc, N_BINS[0], nsb0, w)["snr"] for w in windows_s}})
pd.DataFrame(win_rows).set_index(["night", "observation", "telescope"])

### Per-frame noise floor
Sample-to-sample scatter of the raw 0.5 ms frames (`frame_noise`: differencing removes everything slower than ~1 ms) against the signal level, for stars and dark runs. Each frame's flux proxy is a variance estimated from 2¹⁸ digitizations, which has a relative error of √(2k/2¹⁸) (k = correlated consecutive samples) **independent of the star brightness**. The photon-counting noise model does not include it. A fit σ² = a² + b²/level per telescope separates a level-independent floor `a` from a term that falls with the signal level, like shot noise.

In [ ]:
fr_rows = []
for night in log.nights.values():
    for ch in night.channels:
        for obs in night.observations.values():
            lc = lcs[(night.name, obs.name, ch)]
            fr_rows.append({"night": night.name, "observation": obs.name, "telescope": TELESCOPES[ch].name,
                            "type": obs.type, "level": float(np.median(lc.signal)), "frame_noise": frame_noise(lc)})
frames = pd.DataFrame(fr_rows)
floor_k1 = variance_estimator_floor(config.standard_sampling)

fig, ax = plt.subplots(figsize=(8, 5.5))
fits = {}
for tel_name, g in frames.groupby("telescope"):
    b2, a2 = np.polyfit(1.0 / g["level"], g["frame_noise"] ** 2, 1)
    fits[tel_name] = {"floor_a_pct": 100 * np.sqrt(max(a2, 0)), "b": np.sqrt(max(b2, 0)),
                      "k_equivalent": max(a2, 0) / floor_k1 ** 2}
    pts = ax.scatter(g["level"], 100 * g["frame_noise"], marker=MARKERS[tel_name], label=tel_name)
    lv = np.linspace(g["level"].min(), g["level"].max(), 50)
    ax.plot(lv, 100 * np.sqrt(np.maximum(a2 + b2 / lv, 0)), color=pts.get_facecolor()[0], alpha=0.6)
ax.axhline(100 * floor_k1, color="k", ls=":", label="√(2/2¹⁸): variance-estimator floor, k = 1")
ax.set_xscale("log"); ax.set_xlabel("signal level (variance units)"); ax.set_ylabel("per-frame noise [%]")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
plt.show()
pd.DataFrame(fits).T

## 9. Sources of disparity: checklist

Evidence gathered above (✔ = checked in this notebook, ☐ = still to check / needs information you have to add to the log).

**Measurement side**
- ✔ *Estimator choice.* `mean_*`/`peak_*` ignore correlated noise and run ~8 % high from the small-sample bias at 5 ms; the detrended estimator is the fair one (section 5). Its value depends on the detrend window (table in section 8).
- ✔ *Slow drifts* (transparency, tracking, wind): compare `detr_full` vs `detr` and `detr_despiked`/`detr_dcdetrend` (section 7 table); `dc_bad_frac` gives the fraction of each run flagged by DC excursions.
- ✔ *NSB subtraction.* 2026-03-08 has no dark run, so its NSB comes from the DC, converted with the fast/DC ratio, which ignores the electronic pedestal of the variance (section 4). Cross-check: LST-1's DC-histogram NSB on 2026-03-08 (≈13.1–13.6 fast units) agrees with the 2026-08-19 LST-1 dark patch (13.0). The LST-1 background pixel reads far too high (`mean_fast_dcnsb` goes negative), so use `dc_hist` for LST-1, but only when the DC history has no camera-off periods. On 2026-08-19 its first peak is the closed-camera level (≈0.8 fast units vs 13.0 from the dark patch). 2026-08-19 uses its dark patches.
- ☐ *Good-time windows* for 2026-08-19 are not set (whole runs used).
- ✔ *Correlated (red) noise.* This is the dominant effect. After a 10 s detrend, the bin-to-bin noise is nearly flat at ~1 % from 1 ms to 1 s (section 8); the PSD puts 1–4 % RMS in 0.1–1 Hz against ≲0.5 % in 1–10 Hz. Averaging therefore stops helping after a few ms, and the 100 ms ratios are far worse than the 5 ms ones. Candidate causes: wind-driven structure or mirror vibrations moving the star image across the 0.1° pixel, tracking jitter, and scintillation larger than the Young model.
- ✔ *Per-frame floor.* Even at 0.5 ms the MAGIC scatter has a level-independent floor of ≈0.4 % (fit: 0.38 % for MAGIC-2, 0.42 % for MAGIC-1, i.e. ≈1.4–1.5× √(2/2¹⁸), k ≈ 2). That is consistent with the statistical error of the per-frame variance estimate with ~2 correlated digitizer samples, and it is not in `noise_budget`. LST-1 shows no such floor: its per-frame scatter is level-dependent. ☐ Measuring the digitizer autocorrelation (k) would turn this into a model term.

**Atmosphere**
- ✔ *Extinction at the real Zd* is now in `theo_obs` and `*_ext`. Old `zd_dict` values were wrong for 2026-03-08 (HD-40626 31.3° not 38°, HD-46016 28.8° not 30°).
- ✔ *Clouds.* **2025-12-16: the pyrometer reads 100 % cloudiness all night, with a sky temperature ≈ air temperature (~275 K, vs ~208 K on the clear 2026-03-08).** That is the signature of an overcast sky, or of a covered/faulty pyrometer. Yet the MAGIC-2 starguider identifies 15–32 stars during the runs (fewer than the 18–48 of 2026-03-08), and the measured/theory ratio is not lower that night. Thin or broken cloud, or a pyrometer problem, are both possible. ☐ Check the eLog.
- ✔ *Humidity/wind* (section 2): 2025-12-16 ~55–70 % RH, 2026-03-08 ~25 % RH.
- ☐ *LIDAR:* not operating on any night (ERROR_CODE 4), so there is no measured transmission.
- ✔ *Moon:* below the horizon for all star runs (`moon_alt_deg`).
- ☐ *Scintillation:* the model uses Young's formula with `scint_coeff=1`; La Palma studies suggest ~1.5.

**Telescope**
- ✔ *Faulty mirrors:* 2026-03-08 AMC `PANELS[1]` grows from 7 to 18 on MAGIC-2 during the three runs. ☐ Confirm what the four AMC `PANELS` counts mean, and the panel totals in `snr_measurement.N_PANELS`; only the YAML panel lists enter `theo_obs` for now.
- ☐ *Faulty mirrors 2025-12-16 / 2026-08-19 and LST-1:* unknown.
- ☐ *Star centring in the pixel / PSF:* the starguider pointing deviation (`starg_dev_rms_deg`) is in the conditions table; a star off the pixel centre loses light.
- ☐ *Pixel gain/HV and the PMT excess-noise factor* (model: F = 1.15, 0.7 p.e. electronic noise).

**Model side**
- ✔ *Band:* B and V both plotted; the model's Vega normalisation over the unfiltered QE band is neither.
- ☐ *MAGIC optics* use the LST QE/reflectivity files (see `magic-is-scaled-lst-optics`), so only the area differs.